In [4]:
!pip -q install -U anthropic datasets pandas tqdm tenacity scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.


In [5]:
import os
from getpass import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")
print("Anthropic API key loaded securely.")

Enter your Anthropic API key: ··········
Anthropic API key loaded securely.


In [6]:
from anthropic import Anthropic, BadRequestError, AuthenticationError
import os

client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

MODEL = "claude-opus-4-7"

try:
    message = client.messages.create(
        model=MODEL,
        max_tokens=100,
        messages=[
            {
                "role": "user",
                "content": 'Return JSON only: {"label":"test"}'
            }
        ]
    )

    print("Claude API test succeeded.")
    print(message.content[0].text)

except AuthenticationError as e:
    print("AuthenticationError:")
    print(e)

except BadRequestError as e:
    print("BadRequestError:")
    print(e)

Claude API test succeeded.
{"label":"test"}


In [7]:
import os
import json
import time
import re
import pandas as pd

from json import JSONDecodeError
from tqdm import tqdm
from collections import Counter
from datasets import load_dataset

from anthropic import (
    Anthropic,
    RateLimitError,
    APIConnectionError,
    BadRequestError,
    AuthenticationError
)

from tenacity import (
    retry,
    wait_exponential,
    stop_after_attempt,
    retry_if_exception_type
)

client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

MODEL = "claude-opus-4-7"
DATASET_NAME = "ChanceFocus/flare-ner"

# Evaluate 100 samples.
# The test set has 98 samples, so it will evaluate all 98.
LIMIT = 100

dataset = load_dataset(DATASET_NAME)

split_name = "test" if "test" in dataset else list(dataset.keys())[0]
split_data = dataset[split_name]

print(dataset)
print("Split:", split_name)
print("Columns:", split_data.column_names)
print("Example:", split_data[0])

VALID_LABELS = [
    "O",
    "B-PER", "I-PER",
    "B-ORG", "I-ORG",
    "B-LOC", "I-LOC"
]

SYSTEM_PROMPT = """
You are a financial named entity recognition model.

Task:
Given a list of tokens from a financial text, assign exactly one BIO label to each token.

Allowed labels:
- O
- B-PER, I-PER
- B-ORG, I-ORG
- B-LOC, I-LOC

Definitions:
- PER: person or role-like legal party
- ORG: organization
- LOC: location
- O: not a named entity

Rules:
1. Return exactly one label for each input token.
2. The number of output labels must equal the number of input tokens.
3. Use only the allowed labels.
4. Do not add explanations.
5. Return only valid JSON in this exact format:
{"labels": ["O", "B-ORG", "I-ORG"]}
"""

def get_column_name(example, possible_names):
    for name in possible_names:
        if name in example:
            return name

    raise ValueError(
        f"Cannot find any column from {possible_names}. "
        f"Available columns are: {list(example.keys())}"
    )

def normalize_label(label):
    label = str(label).strip()

    if label in VALID_LABELS:
        return label

    if label in ["PER_B", "ORG_B", "LOC_B"]:
        return "B-" + label.split("_")[0]

    if label in ["PER_I", "ORG_I", "LOC_I"]:
        return "I-" + label.split("_")[0]

    return "O"

def get_tokens(row, token_col=None):
    if token_col is not None and token_col in row:
        return row[token_col]

    if "text" in row and row["text"] is not None:
        return str(row["text"]).split()

    raise ValueError("Cannot find tokens or text.")

def parse_claude_json(raw_text):
    raw_text = str(raw_text).strip()

    if raw_text == "":
        raise ValueError("Empty model output")

    try:
        return json.loads(raw_text)
    except JSONDecodeError:
        match = re.search(r"\{.*\}", raw_text, flags=re.DOTALL)
        if match:
            return json.loads(match.group(0))

        print("Raw model output:")
        print(repr(raw_text))
        raise

def fix_prediction_length(pred_labels, target_len):
    pred_labels = [normalize_label(x) for x in pred_labels]

    if len(pred_labels) < target_len:
        pred_labels = pred_labels + ["O"] * (target_len - len(pred_labels))

    if len(pred_labels) > target_len:
        pred_labels = pred_labels[:target_len]

    return pred_labels

def bio_to_spans(labels):
    spans = []
    start = None
    ent_type = None

    for i, label in enumerate(labels + ["O"]):
        if label == "O":
            if start is not None:
                spans.append((start, i, ent_type))
                start = None
                ent_type = None
            continue

        if "-" not in label:
            continue

        prefix, label_type = label.split("-", 1)

        if prefix == "B":
            if start is not None:
                spans.append((start, i, ent_type))
            start = i
            ent_type = label_type

        elif prefix == "I":
            if start is None:
                start = i
                ent_type = label_type
            elif ent_type != label_type:
                spans.append((start, i, ent_type))
                start = i
                ent_type = label_type

    return spans

def compute_metrics(gold_all, pred_all):
    correct_tokens = 0
    total_tokens = 0
    exact_match_count = 0

    tp = 0
    fp = 0
    fn = 0

    for gold_labels, pred_labels in zip(gold_all, pred_all):
        total_tokens += len(gold_labels)
        correct_tokens += sum(g == p for g, p in zip(gold_labels, pred_labels))

        if gold_labels == pred_labels:
            exact_match_count += 1

        gold_spans = Counter(bio_to_spans(gold_labels))
        pred_spans = Counter(bio_to_spans(pred_labels))

        tp += sum((gold_spans & pred_spans).values())
        fp += sum((pred_spans - gold_spans).values())
        fn += sum((gold_spans - pred_spans).values())

    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    token_accuracy = correct_tokens / total_tokens if total_tokens > 0 else 0.0
    exact_match_accuracy = exact_match_count / len(gold_all) if len(gold_all) > 0 else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "token_accuracy": token_accuracy,
        "exact_match_accuracy": exact_match_accuracy,
        "true_positive": tp,
        "false_positive": fp,
        "false_negative": fn
    }

@retry(
    retry=retry_if_exception_type((RateLimitError, APIConnectionError, JSONDecodeError, ValueError)),
    wait=wait_exponential(multiplier=5, min=10, max=120),
    stop=stop_after_attempt(5),
    reraise=True
)
def call_claude_ner(tokens):
    indexed_tokens = "\n".join([f"{i}: {tok}" for i, tok in enumerate(tokens)])

    user_prompt = (
        f"Number of tokens: {len(tokens)}\n\n"
        f"Tokens:\n{indexed_tokens}\n\n"
        f"Return exactly {len(tokens)} BIO labels as JSON."
    )

    try:
        message = client.messages.create(
            model=MODEL,
            max_tokens=1500,
            system=SYSTEM_PROMPT,
            messages=[
                {
                    "role": "user",
                    "content": user_prompt
                }
            ]
        )

    except BadRequestError as e:
        print("\nBadRequestError detail:")
        print(e)
        raise

    except AuthenticationError as e:
        print("\nAuthenticationError detail:")
        print(e)
        raise

    raw_output = "".join(
        block.text for block in message.content
        if getattr(block, "type", None) == "text"
    )

    parsed = parse_claude_json(raw_output)

    if "labels" not in parsed:
        raise ValueError(f"Missing labels field in model output: {parsed}")

    return parsed

example = split_data[0]

token_col = None
if "token" in example:
    token_col = "token"
elif "tokens" in example:
    token_col = "tokens"

label_col = get_column_name(example, ["label", "labels", "tags"])

print("Token column:", token_col if token_col else "text.split()")
print("Label column:", label_col)

sample_size = min(LIMIT, len(split_data))
eval_data = split_data.select(range(sample_size))

rows = []
gold_all = []
pred_all = []

for idx, row in enumerate(tqdm(eval_data)):
    tokens = get_tokens(row, token_col)
    gold_labels = [normalize_label(x) for x in row[label_col]]

    if len(tokens) != len(gold_labels) and "text" in row:
        tokens = str(row["text"]).split()

    try:
        result = call_claude_ner(tokens)
        pred_labels = result.get("labels", [])
        pred_labels = fix_prediction_length(pred_labels, len(gold_labels))
        error = ""

    except BadRequestError as e:
        print(f"\nStopping at row {idx} because of BadRequestError.")
        print(e)
        raise

    except AuthenticationError as e:
        print(f"\nStopping at row {idx} because of AuthenticationError.")
        print(e)
        raise

    except Exception as e:
        pred_labels = ["O"] * len(gold_labels)
        error = str(e)
        print(f"Error at row {idx}: {e}")

    gold_all.append(gold_labels)
    pred_all.append(pred_labels)

    rows.append({
        "idx": idx,
        "id": row.get("id", ""),
        "text": row.get("text", ""),
        "tokens": tokens,
        "gold_labels": gold_labels,
        "pred_labels": pred_labels,
        "gold_spans": bio_to_spans(gold_labels),
        "pred_spans": bio_to_spans(pred_labels),
        "exact_match": gold_labels == pred_labels,
        "error": error
    })

    time.sleep(5)

metrics = compute_metrics(gold_all, pred_all)

print("\n===== Claude Opus 4.7 FLARE-NER Evaluation =====")
print(f"Dataset: {DATASET_NAME}")
print(f"Split: {split_name}")
print(f"Model: {MODEL}")
print(f"Samples evaluated: {len(eval_data)}")
print(f"Precision / Correctness Rate: {metrics['precision']:.4f}")
print(f"Recall: {metrics['recall']:.4f}")
print(f"F1 Score: {metrics['f1_score']:.4f}")
print(f"Exact Match Accuracy: {metrics['exact_match_accuracy']:.4f}")
print(f"Token Accuracy: {metrics['token_accuracy']:.4f}")
print(f"TP: {metrics['true_positive']}")
print(f"FP: {metrics['false_positive']}")
print(f"FN: {metrics['false_negative']}")

df = pd.DataFrame(rows)

df.to_csv("Claude_Opus_4_7_FLARE_NER_Evaluation_Results.csv", index=False)

with open("Claude_Opus_4_7_FLARE_NER_Metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

num_errors = df["error"].astype(str).str.len().gt(0).sum()

print("\n===== Error Summary =====")
print(f"Failed samples: {num_errors}")
print(f"Total samples: {len(df)}")
print(f"Failure rate: {num_errors / len(df):.4f}")

pd.set_option("display.max_colwidth", 120)
df.head()

DatasetDict({
    train: Dataset({
        features: ['query', 'answer', 'label', 'text'],
        num_rows: 408
    })
    test: Dataset({
        features: ['query', 'answer', 'label', 'text'],
        num_rows: 98
    })
    valid: Dataset({
        features: ['query', 'answer', 'label', 'text'],
        num_rows: 103
    })
})
Split: test
Columns: ['query', 'answer', 'label', 'text']
Example: {'query': "In the sentences extracted from financial agreements in U.S. SEC filings, identify the named entities that represent a person ('PER'), an organization ('ORG'), or a location ('LOC'). The required answer format is: 'entity name, entity type'.\nText: Subordinated Loan Agreement - Silicium de Provence SAS and Evergreen Solar Inc . 7 - December 2007 [ HERBERT SMITH LOGO ] ................................ 2007 SILICIUM DE PROVENCE SAS and EVERGREEN SOLAR , INC .\nAnswer:", 'answer': 'Silicium de Provence SAS, ORG\nEvergreen Solar Inc, ORG\nHERBERT SMITH, PER\nSILICIUM DE PROVENCE SAS, OR

100%|██████████| 98/98 [13:10<00:00,  8.07s/it]


===== Claude Opus 4.7 FLARE-NER Evaluation =====
Dataset: ChanceFocus/flare-ner
Split: test
Model: claude-opus-4-7
Samples evaluated: 98
Precision / Correctness Rate: 0.2768
Recall: 0.1576
F1 Score: 0.2008
Exact Match Accuracy: 0.1327
Token Accuracy: 0.9378
TP: 49
FP: 128
FN: 262

===== Error Summary =====
Failed samples: 0
Total samples: 98
Failure rate: 0.0000


,idx,id,text,tokens,gold_labels,pred_labels,gold_spans,pred_spans,exact_match,error
0,0,,Subordinated Loan Agreement - Silicium de Provence SAS and Evergreen Solar Inc . 7 - December 2007 [ HERBERT SMITH L...,"[Subordinated, Loan, Agreement, -, Silicium, de, Provence, SAS, and, Evergreen, Solar, Inc, ., 7, -, December, 2007,...","[O, O, O, O, B-ORG, I-ORG, I-ORG, I-ORG, O, B-ORG, I-ORG, I-ORG, O, O, O, O, O, O, B-PER, I-PER, O, O, O, O, B-ORG, ...","[O, O, O, O, B-ORG, I-ORG, I-ORG, I-ORG, O, B-ORG, I-ORG, I-ORG, I-ORG, O, O, O, O, O, B-ORG, I-ORG, I-ORG, O, O, O,...","[(4, 8, ORG), (9, 12, ORG), (18, 20, PER), (24, 28, ORG), (29, 31, ORG)]","[(4, 8, ORG), (9, 13, ORG), (18, 21, ORG), (24, 28, ORG), (29, 34, ORG)]",False,
1,1,,SUBORDINATED LOAN AGREEMENT HERBERT SMITH LLP Page 1 of 12 7 - December 2007 TABLE OF CONTENTS Clause Headings Page 1 .,"[SUBORDINATED, LOAN, AGREEMENT, HERBERT, SMITH, LLP, Page, 1, of, 12, 7, -, December, 2007, TABLE, OF, CONTENTS, Cla...","[O, O, O, B-PER, I-PER, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[O, O, O, B-ORG, I-ORG, I-ORG, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[(3, 5, PER)]","[(3, 6, ORG)]",False,
2,2,,"DISPUTES 10 Page 2 of 12 7 - December 2007 SUBORDINATED LOAN AGREEMENT THIS LOAN AGREEMENT is made on 7th December ,...","[DISPUTES, 10, Page, 2, of, 12, 7, -, December, 2007, SUBORDINATED, LOAN, AGREEMENT, THIS, LOAN, AGREEMENT, is, made...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, B-ORG, I-ORG, I-ORG, O, O, O, O...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, B-ORG, I-ORG, I-ORG, I-ORG, I-O...","[(28, 31, ORG), (49, 50, LOC), (57, 61, LOC), (62, 66, LOC), (67, 68, LOC), (73, 75, PER), (82, 83, PER), (88, 92, O...","[(28, 37, ORG), (49, 50, LOC), (57, 61, LOC), (62, 64, LOC), (64, 66, LOC), (67, 68, LOC), (71, 73, PER), (73, 75, P...",False,
3,3,,WHEREAS : ( A ) The Borrower intends to develop a plant in France for the production of solar grade silicon .,"[WHEREAS, :, (, A, ), The, Borrower, intends, to, develop, a, plant, in, France, for, the, production, of, solar, gr...","[O, O, O, O, O, O, B-PER, O, O, O, O, O, O, B-LOC, O, O, O, O, O, O, O, O]","[O, O, O, O, O, O, O, O, O, O, O, O, O, B-LOC, O, O, O, O, O, O, O, O]","[(6, 7, PER), (13, 14, LOC)]","[(13, 14, LOC)]",False,
4,4,,( B ) Lender and Borrower have entered into an agreement for the sale and purchase of solar grade silicon on the sam...,"[(, B, ), Lender, and, Borrower, have, entered, into, an, agreement, for, the, sale, and, purchase, of, solar, grade...","[O, O, O, B-PER, O, B-PER, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[(3, 4, PER), (5, 6, PER)]",[],False,
